# Criba de motores TTS — OmniVoice, prueba de acento

¿Se puede rebajar el acento inglés de OmniVoice sin entrenar? Repite **10 frases de Naval** con tres ajustes y las deja junto a las del cuaderno 02 para comparar el mismo item:

| Variante | Referencia | `guidance_scale` |
|---|---|---|
| `omnivoice` (ya hecha, cuaderno 02) | `naval_ref02` (~12 s) | 2,0 (por defecto) |
| `omnivoice_ref01` | `naval_ref01` (~6 s) | 2,0 |
| `omnivoice_g3` | `naval_ref02` | 3,0 |
| `omnivoice_ref01_g3` | `naval_ref01` | 3,0 |

Por qué: OmniVoice recomienda referencias de 3–10 s; cuanto más larga, más forma de hablar inglesa copia. Una guía más alta lo ciñe más al texto y a la etiqueta de idioma `es`.

**GPU:** L4 o T4. **Tiempo estimado:** unos 15 min, incluida la instalación.

**Cómo se usa** (es igual en todos los cuadernos de la criba):

1. *Entorno de ejecución → Cambiar tipo de entorno* → **L4 o T4**.
2. Ejecuta la **celda 1** (instalación). Cuando termine, *Entorno de ejecución → Reiniciar sesión*.
3. Ejecuta las demás celdas **en orden**, de la 2 a la última. La celda 2 comprueba que estén las
   referencias y las frases antes de cargar ningún modelo: si dice `RESULTADO: REVISAR`, para ahí.
4. La última celda **desconecta la GPU sola** para que no siga gastando unidades.

Si la sesión se corta a medias, vuelve a empezar por la celda 1: lo que ya esté en Drive no se
genera otra vez.

Salida: `fine_tunned_naval/outputs/criba/<variante>/` en Drive, un `.wav` por item (`n01`, `v01`,
`b01`, `k01`…) más `registro.jsonl`, `parametros.json` y `entorno.json`.

In [ ]:
# 1. Instalación (después: Entorno de ejecución → Reiniciar sesión)
!pip install -q omnivoice==0.2.1

In [ ]:
# 2. Preparar: código del repo, Drive, frases y las dos referencias de Naval
import os, sys, subprocess
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path.insert(0, "/content/repo/criba")
import comun

comun.montar_drive()
TODOS, SILENCIO_MS = comun.cargar_items(["naval"])
ESCOGIDOS = ["n01", "n03", "n05", "n07", "n10", "n12", "n14", "n17", "n21", "n25"]
ITEMS = [it for it in TODOS if it["id"] in ESCOGIDOS]
REF02 = comun.comprobar(ITEMS, slot="ref02")
REF01 = comun.comprobar(ITEMS, slot="ref01")

In [ ]:
# 3. Cargar el modelo
import torch
from pathlib import Path
from omnivoice import OmniVoice
modelo = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16)
SR = modelo.sampling_rate or 24000

def hacer_sintetizar(guia):
    def sintetizar(texto, ref):
        audios = modelo.generate(text=texto, language="es", ref_audio=ref["wav"], ref_text=ref["texto"],
                                 guidance_scale=guia)
        return audios[0], SR
    return sintetizar
print("OK, sale a", SR, "Hz")

In [ ]:
# 4. Generar las tres variantes
VARIANTES = [("omnivoice_ref01", REF01, 2.0), ("omnivoice_g3", REF02, 3.0), ("omnivoice_ref01_g3", REF01, 3.0)]
for variante, refs, guia in VARIANTES:
    print(f"\n=== {variante}")
    comun.guardar_entorno(variante, ["omnivoice", "torch", "transformers"])
    comun.generar(variante, hacer_sintetizar(guia), ITEMS, refs, SILENCIO_MS,
                  params={"modelo": "k2-fsa/OmniVoice", "language": "es", "dtype": "float16",
                          "guidance_scale": guia, "ref": Path(refs["naval"]["wav"]).name})

In [ ]:
# Última celda: resumen y desconexión de la GPU
import json
for variante in ['omnivoice_ref01', 'omnivoice_g3', 'omnivoice_ref01_g3']:
    reg = comun.SALIDA / variante / "registro.jsonl"
    if not reg.exists():
        print(f"{variante}: sin registro")
        continue
    filas = [json.loads(l) for l in reg.read_text(encoding="utf-8").splitlines()]
    ok = sum(f["ok"] for f in filas)
    print(f"{variante}: {ok} items correctos, {len(filas) - ok} con error")
comun.desconectar()